# Assignment 2 (Modul 3) - First-Class Function & Data Processing
## Sistem Pencatatan Data Eksperimen Kecepatan Internet
**Nama:** Farid Al Farizi
**NIM:** 202410370110017

Fitur baru: **Analisis Kualitas Koneksi**, yaitu memberi label kualitas pada setiap hasil speedtest, menyaring data, menghitung total/rata-rata, dan mencari lokasi tercepat.

## 1. Dataset (lanjutan dari Modul 2)

In [1]:
# (NIM: 202410370110017)

from functools import reduce
import copy

def _t(i, owner, loc, conn, dl, ul, ping, stable):
    return {"test_id": i, "owner": owner, "location": loc, "connection": conn,
            "download_mbps": dl, "upload_mbps": ul, "ping_ms": ping, "stable": stable}

SPEED_TESTS = [
    _t(1,  "farid17", "Lab Informatika UMM",   "WiFi",    48.5, 20.3, 14, True),
    _t(2,  "nayla02", "Kos Tlogomas",          "WiFi",    22.7,  8.1, 31, False),
    _t(3,  "dika05",  "Alun-Alun Batu",        "Seluler", 15.2,  5.6, 45, True),
    _t(4,  "farid17", "Perpustakaan UMM",      "WiFi",    35.9, 15.2, 18, True),
    _t(5,  "farid17", "Rumah Jember",          "LAN",     95.4, 40.1,  9, True),
    _t(6,  "rani08",  "Cafe Dinoyo",           "WiFi",    12.3,  4.4, 52, False),
    _t(7,  "bima11",  "Stasiun Malang",        "Seluler", 28.6, 10.9, 38, True),
    _t(8,  "salsa03", "GOR UMM",               "Seluler",  9.8,  3.1, 67, False),
    _t(9,  "yoga21",  "Kos Landungsari",       "WiFi",    41.0, 18.7, 21, True),
    _t(10, "intan09", "Lab Jaringan UMM",      "LAN",     88.2, 37.5, 11, True),
    _t(11, "farid17", "Desa Petungsewu",       "Seluler",  6.4,  2.0, 89, False),
    _t(12, "hafiz14", "Masjid AR Fachruddin",  "WiFi",    30.5, 12.0, 25, True),
    _t(13, "citra06", "Matos",                 "Seluler", 19.7,  7.3, 40, True),
    _t(14, "rizky19", "Terminal Arjosari",     "Seluler", 11.1,  4.9, 58, False),
    _t(15, "laras12", "Kos Tegalgondo",        "WiFi",    26.4,  9.6, 29, True),
]

# Simpan salinan untuk membuktikan data asli tidak berubah di akhir
ORIGINAL = copy.deepcopy(SPEED_TESTS)
print(len(SPEED_TESTS), "catatan eksperimen")

15 catatan eksperimen


## 2. First-Class Function & Higher-Order Function
- `classify_speed` adalah fungsi biasa yang menentukan kategori kecepatan.
- `apply_label(classifier)` adalah **HOF**: menerima fungsi sebagai parameter dan mengembalikan fungsi baru.
- `make_filter(field, minimum)` adalah **HOF**: mengembalikan fungsi predikat yang siap dipakai di `filter()`.

In [5]:
# (NIM: 202410370110017)

def classify_speed(download):
    #Cepat >= 40 Mbps, Sedang >= 15 Mbps, selain itu Lambat.
    if download >= 40:
        return "Cepat"
    if download >= 15:
        return "Sedang"
    return "Lambat"


def apply_label(classifier):
    #HOF
    def add_label(test):
        return {**test, "quality": classifier(test["download_mbps"])}   # dict baru, data asli aman
    return add_label


def make_filter(field, minimum):
    #HOF
    def check(test):
        return test[field] >= minimum
    return check


# Fungsi disimpan di variabel (first-class)
label_quality = apply_label(classify_speed)
is_fast_download = make_filter("download_mbps", 25)

print(label_quality(SPEED_TESTS[0]))
print(is_fast_download(SPEED_TESTS[0]), is_fast_download(SPEED_TESTS[1]))

{'test_id': 1, 'owner': 'farid17', 'location': 'Lab Informatika UMM', 'connection': 'WiFi', 'download_mbps': 48.5, 'upload_mbps': 20.3, 'ping_ms': 14, 'stable': True, 'quality': 'Cepat'}
True False


## 3. Pemrosesan Data: `filter()`, `map()`, `reduce()`

In [ ]:
# (NIM: 202410370110017)

def is_owned_by(username):
    #HOF
    def check(test):
        return test["owner"] == username
    return check


def is_stable(test):
    return test["stable"]


def get_download(test):
    return test["download_mbps"]


def add(a, b):
    return a + b


# filter(): ambil data milik farid17 saja
my_tests = list(filter(is_owned_by("farid17"), SPEED_TESTS))

# map(): beri label kualitas pada setiap catatan
labeled = list(map(label_quality, my_tests))
for t in labeled:
    print(f"#{t['test_id']} {t['location']:<20} {t['download_mbps']:>5} Mbps -> {t['quality']}")

# reduce(): jumlahkan seluruh download, lalu hitung rata-rata
total_download = reduce(add, map(get_download, my_tests), 0)
print(f"\nTotal download   : {total_download:.1f} Mbps")
print(f"Rata-rata download: {total_download / len(my_tests):.2f} Mbps")

#1 Lab Informatika UMM   48.5 Mbps -> Cepat
#4 Perpustakaan UMM      35.9 Mbps -> Sedang
#5 Rumah Jember          95.4 Mbps -> Cepat
#11 Desa Petungsewu        6.4 Mbps -> Lambat

Total download   : 186.2 Mbps
Rata-rata download: 46.55 Mbps


## 4. Recursion

In [ ]:
# (NIM: 202410370110017)

def find_fastest(tests):
    #Rekursif: mencari catatan dengan download tertinggi.
    if len(tests) == 1:                      # base case: tinggal satu data
        return tests[0]
    best_of_rest = find_fastest(tests[1:])   # recursive case: cari di sisa list (tail)
    return tests[0] if tests[0]["download_mbps"] >= best_of_rest["download_mbps"] else best_of_rest


def count_if(tests, predicate):
    #Rekursif + HOF: menghitung jumlah data yang lolos predicate.
    if not tests:                            # base case: list kosong
        return 0
    head = 1 if predicate(tests[0]) else 0
    return head + count_if(tests[1:], predicate)


fastest = find_fastest(my_tests)
print(f"Lokasi tercepat : {fastest['location']} ({fastest['download_mbps']} Mbps)")
print(f"Tes stabil      : {count_if(my_tests, is_stable)} dari {len(my_tests)}")

Lokasi tercepat : Rumah Jember (95.4 Mbps)
Tes stabil      : 3 dari 4


## 5. Fitur Lengkap: Laporan Analisis Kualitas Koneksi

In [6]:
# (NIM: 202410370110017)

def is_quality(quality):
    #HOF: predikat untuk memilih catatan dengan kualitas tertentu.
    def check(test):
        return test["quality"] == quality
    return check


def quality_report(tests, username):
    #Menggabungkan filter, map, reduce, dan rekursi menjadi satu laporan (pure, tanpa print).
    mine = list(filter(is_owned_by(username), tests))
    if not mine:
        return f"--- Analisis {username} ---\n  Belum ada data."
    labeled = list(map(label_quality, mine))
    total = reduce(add, map(get_download, mine), 0)
    fastest = find_fastest(mine)
    lines = [
        f"--- Analisis Kualitas Koneksi {username} ---",
        f"  Jumlah tes        : {len(mine)}",
        f"  Rata-rata download: {total / len(mine):.2f} Mbps",
        f"  Cepat / Sedang / Lambat: {count_if(labeled, is_quality('Cepat'))} / "
        f"{count_if(labeled, is_quality('Sedang'))} / {count_if(labeled, is_quality('Lambat'))}",
        f"  Tes stabil        : {count_if(mine, is_stable)}",
        f"  Lokasi tercepat   : {fastest['location']} ({fastest['download_mbps']} Mbps)",
    ]
    return "\n".join(lines)


print(quality_report(SPEED_TESTS, "farid17"))
print()
print(quality_report(SPEED_TESTS, "nayla02"))
print()
print(quality_report(SPEED_TESTS, "tidak_ada"))

--- Analisis Kualitas Koneksi farid17 ---
  Jumlah tes        : 4
  Rata-rata download: 46.55 Mbps
  Cepat / Sedang / Lambat: 2 / 1 / 1
  Tes stabil        : 3
  Lokasi tercepat   : Rumah Jember (95.4 Mbps)

--- Analisis Kualitas Koneksi nayla02 ---
  Jumlah tes        : 1
  Rata-rata download: 22.70 Mbps
  Cepat / Sedang / Lambat: 0 / 1 / 0
  Tes stabil        : 0
  Lokasi tercepat   : Kos Tlogomas (22.7 Mbps)

--- Analisis tidak_ada ---
  Belum ada data.


## 6. Verifikasi Immutability

In [7]:
# (NIM: 202410370110017)

print("SPEED_TESTS tidak berubah:", SPEED_TESTS == ORIGINAL)
print("Ada key 'quality' di data asli?", "quality" in SPEED_TESTS[0])

SPEED_TESTS tidak berubah: True
Ada key 'quality' di data asli? False


## 7. Alasan Penggunaan Konsep

| Konsep / Fungsi | Dipakai di | Alasan |
|---|---|---|
| **First-Class Function** | `label_quality`, `is_fast_download` disimpan di variabel | Fungsi bisa diperlakukan seperti data sehingga mudah dikirim ke `map()`/`filter()`. |
| **HOF (menerima fungsi)** | `apply_label(classifier)`, `count_if(tests, predicate)` | Cara memberi label atau kriteria hitungan bisa diganti tanpa mengubah kode fungsi utama. |
| **HOF (mengembalikan fungsi)** | `make_filter`, `is_owned_by`, `is_quality` | Membuat predikat baru cukup dengan argumen berbeda (closure), tidak perlu menulis banyak fungsi serupa. |
| **`filter()`** | Mengambil catatan milik user tertentu | Hanya sebagian data yang dibutuhkan; `filter()` menyaring tanpa mengubah list asli. |
| **`map()`** | Menambahkan label kualitas dan mengambil nilai download | Setiap catatan perlu ditransformasi dengan cara yang sama; panjang hasil tetap sama. |
| **`reduce()`** | Menjumlahkan seluruh download | Banyak nilai perlu dilipat menjadi satu angka total. Initializer `0` mencegah error saat data kosong. |
| **Recursion** | `find_fastest`, `count_if` | Menggantikan loop dengan pola head/tail. Base case menghentikan rekursi, dan tidak ada counter yang dimutasi. |

Semua fungsi bersifat **pure**: `apply_label` membuat dict baru dengan `{**test, ...}`, sehingga `SPEED_TESTS` tetap utuh (dibuktikan di bagian 6).